# LogReg + LASSO — Búsqueda de Hiperparámetros con Optuna (L1, L2, L3)

**Objetivo:** equiparar el tuning del baseline (LogReg+LASSO) al esfuerzo aplicado a CatBoost para garantizar una comparación metodológicamente justa entre arquitecturas.

**Features:** idénticas al notebook `01_baseline_logreg.ipynb` (8 numéricas + TF-IDF sobre `chiefcomplaint`). El `max_features` del TF-IDF es uno de los hiperparámetros buscados.

**Estrategia:** Optuna con `TPESampler(seed=42)`, 30 trials por target (mismo presupuesto que CatBoost), maximizando AUROC sobre validación temporal.

**Tiempo estimado:** 20-40 min total (LogReg es computacionalmente ligero sobre 278k filas).

## 1. Setup y librerías

Importaciones, configuración de logging y constantes globales del experimento. `N_TRIALS=30` se fija para equiparar el presupuesto de búsqueda con CatBoost.

In [1]:
import os
import json
import logging
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.sparse as sp
import joblib
import optuna
from dotenv import load_dotenv

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

optuna.logging.set_verbosity(optuna.logging.WARNING)
logging.getLogger('sklearn').setLevel(logging.ERROR)
import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings('ignore', category=ConvergenceWarning)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 150, 'font.size': 10})

N_TRIALS = 30
TARGETS = ['L1', 'L2', 'L3']
TARGET_NAMES = {
    'L1': 'Ingreso hospitalario',
    'L2': 'Resultado crítico en urgencias',
    'L3': 'Intervención crítica en urgencias',
}
RANDOM_SEED = 42
print(f'Optuna {optuna.__version__} — N_TRIALS={N_TRIALS} por target')

Optuna 4.8.0 — N_TRIALS=30 por target


## 2. Carga de datos y feature engineering base

Se cargan las particiones temporales y se construyen las matrices de features (numéricas + TF-IDF) usando el mismo procedimiento que en el notebook baseline `01_baseline_logreg.ipynb`. El ajuste de imputación y escalado se realiza exclusivamente sobre train.

In [2]:
load_dotenv(dotenv_path=Path('../../.env'), override=True)
if not os.getenv('MIMIC_IV_ED_PATH'):
    load_dotenv(dotenv_path=Path('.env'), override=True)

DATA = Path(os.getenv('MIMIC_IV_ED_PATH', ''))
PROCESSED_DIR = Path('../../data/processed')

print(f'DATA         : {DATA}  |  existe: {DATA.exists()}')
print(f'PROCESSED_DIR: {PROCESSED_DIR.resolve()}')

df_train = pd.read_parquet(PROCESSED_DIR / 'train.parquet')
df_val   = pd.read_parquet(PROCESSED_DIR / 'val.parquet')
print(f'\nTrain: {df_train.shape[0]:,} × {df_train.shape[1]}  |  Val: {df_val.shape[0]:,} × {df_val.shape[1]}')

DATA         : C:\Users\cuent\Documents\UAX\TFM\TFM\CodigoGit\data  |  existe: True
PROCESSED_DIR: C:\Users\cuent\Documents\UAX\TFM\TFM\CodigoGit\data\processed

Train: 278,320 × 21  |  Val: 59,640 × 21


In [3]:
load_dotenv(dotenv_path=Path('../../.env'), override=True)
if not os.getenv('MIMIC_IV_ED_PATH'):
    load_dotenv(dotenv_path=Path('.env'), override=True)

DATA = Path(os.getenv('MIMIC_IV_ED_PATH', ''))
PROCESSED_DIR = Path('../../data/processed')

print(f'DATA         : {DATA}  |  existe: {DATA.exists()}')
print(f'PROCESSED_DIR: {PROCESSED_DIR.resolve()}')

df_train = pd.read_parquet(PROCESSED_DIR / 'train.parquet')
df_val   = pd.read_parquet(PROCESSED_DIR / 'val.parquet')
print(f'\nTrain: {df_train.shape[0]:,} × {df_train.shape[1]}  |  Val: {df_val.shape[0]:,} × {df_val.shape[1]}')

# Agregación de n_medications desde medrecon
print('Cargando medrecon...')
df_medrecon = pd.read_csv(DATA / 'medrecon.csv', low_memory=False)
n_meds = df_medrecon.groupby('stay_id').size().rename('n_medications').reset_index()

df_train = df_train.merge(n_meds, on='stay_id', how='left')
df_val   = df_val.merge(n_meds, on='stay_id', how='left')
df_train['n_medications'] = df_train['n_medications'].fillna(0).astype(int)
df_val['n_medications']   = df_val['n_medications'].fillna(0).astype(int)

# pain se almacena como str en los parquets; coerción a float antes de imputación
df_train['pain'] = pd.to_numeric(df_train['pain'], errors='coerce')
df_val['pain']   = pd.to_numeric(df_val['pain'],   errors='coerce')

NUMERIC_FEATURES = ['temperature', 'heartrate', 'resprate', 'o2sat', 'sbp', 'dbp', 'acuity', 'n_medications', 'pain']

# Imputación y escalado ajustados exclusivamente sobre train
imputer = SimpleImputer(strategy='median')
scaler  = StandardScaler()
X_num_train = scaler.fit_transform(imputer.fit_transform(df_train[NUMERIC_FEATURES]))
X_num_val   = scaler.transform(imputer.transform(df_val[NUMERIC_FEATURES]))

# One-hot encoding de arrival_transport (ajustado sobre train)
ohe = OneHotEncoder(sparse_output=True, handle_unknown='ignore')
X_cat_train = ohe.fit_transform(df_train[['arrival_transport']].fillna('desconocido'))
X_cat_val   = ohe.transform(df_val[['arrival_transport']].fillna('desconocido'))

train_text = df_train['chiefcomplaint'].fillna('').astype(str).str.lower()
val_text   = df_val['chiefcomplaint'].fillna('').astype(str).str.lower()

print(f'X_num_train: {X_num_train.shape}  |  X_num_val: {X_num_val.shape}')
print(f'X_cat_train (OHE): {X_cat_train.shape}  |  Textos: train={len(train_text):,}')

DATA         : C:\Users\cuent\Documents\UAX\TFM\TFM\CodigoGit\data  |  existe: True
PROCESSED_DIR: C:\Users\cuent\Documents\UAX\TFM\TFM\CodigoGit\data\processed

Train: 278,320 × 21  |  Val: 59,640 × 21
Cargando medrecon...
X_num_train: (278320, 9)  |  X_num_val: (59640, 9)
X_cat_train (OHE): (278320, 5)  |  Textos: train=278,320


## 3. Función objetivo de Optuna

Espacio de búsqueda (mismo presupuesto que CatBoost, 30 trials):

| Parámetro | Rango | Tipo |
|-----------|-------|------|
| `C` | [1e-3, 1e2] | log-uniform |
| `class_weight` | {balanced, None} | categórico |
| `tfidf_max_features` | [30, 100] | int |

Penalización L1 (LASSO) fija. **Solver fijo a `liblinear`** — coordinate descent, más eficiente que `saga` para LogReg+L1 sobre matrices sparse de tamaño medio (Fan et al., 2008, *LIBLINEAR: A Library for Large Linear Classification*). `max_iter=500` es suficiente para `liblinear`, que converge en menos iteraciones que `saga` con este tipo de datos.

**Nota sobre equidad metodológica:** se mantienen los 30 trials (mismo presupuesto que CatBoost). La restricción del solver es un detalle de implementación de sklearn y no afecta al espacio de hiperparámetros del modelo. Usar `saga` añadía ~2 min por trial sin aporte predictivo adicional (ambos solvers resuelven el mismo problema matemático).

In [4]:
def build_matrix(max_features):
    """Reconstruye la matriz combinada (num + OHE + TF-IDF) con el max_features dado."""
    tfidf = TfidfVectorizer(max_features=max_features, token_pattern=r'(?u)\b\w+\b')
    X_text_train = tfidf.fit_transform(train_text)
    X_text_val   = tfidf.transform(val_text)
    X_tr = sp.hstack([sp.csr_matrix(X_num_train), X_cat_train, X_text_train]).tocsr()
    X_v  = sp.hstack([sp.csr_matrix(X_num_val),   X_cat_val,   X_text_val  ]).tocsr()
    return X_tr, X_v, tfidf


def make_objective(y_tr, y_v):
    def objective(trial):
        C = trial.suggest_float('C', 1e-3, 1e2, log=True)
        class_weight = trial.suggest_categorical('class_weight', ['balanced', None])
        max_features = trial.suggest_int('tfidf_max_features', 30, 100)

        X_tr, X_v, _ = build_matrix(max_features)

        # l1_ratio=1 equivale a penalty='l1'; penalty fue deprecado en sklearn 1.8
        clf = LogisticRegression(
            l1_ratio=1,
            C=C,
            class_weight=class_weight,
            solver='liblinear',
            max_iter=500,
            random_state=RANDOM_SEED,
        )
        clf.fit(X_tr, y_tr)
        y_pred = clf.predict_proba(X_v)[:, 1]
        return roc_auc_score(y_v, y_pred)
    return objective

## 4. Optimización por target

Se ejecuta la búsqueda bayesiana para cada target por separado. Optuna maximiza el AUROC en validación con los parámetros definidos en la función objetivo.

In [5]:
best_params = {}
best_aurocs = {}

for target in TARGETS:
    y_tr = df_train[target]
    y_v  = df_val[target]

    print(f"\n{'='*65}")
    print(f'Optimizando TARGET: {target} — {TARGET_NAMES[target]}')
    print(f'Prevalencia train: {y_tr.mean()*100:.2f}%')
    print(f"{'='*65}")

    sampler = optuna.samplers.TPESampler(seed=RANDOM_SEED)
    study = optuna.create_study(direction='maximize', study_name=f'logreg_{target}', sampler=sampler)
    study.optimize(make_objective(y_tr, y_v), n_trials=N_TRIALS, show_progress_bar=True)

    best_params[target] = study.best_params
    best_aurocs[target] = study.best_value

    print(f'\n{target} — Mejor AUROC (Optuna): {study.best_value:.4f}')
    print(f'   Mejores parámetros: {study.best_params}')

print('\nBúsqueda completada para todos los targets.')


Optimizando TARGET: L1 — Ingreso hospitalario
Prevalencia train: 38.55%


  0%|          | 0/30 [00:00<?, ?it/s]


L1 — Mejor AUROC (Optuna): 0.8281
   Mejores parámetros: {'C': 23.714140027272748, 'class_weight': 'balanced', 'tfidf_max_features': 100}

Optimizando TARGET: L2 — Resultado crítico en urgencias
Prevalencia train: 1.54%


  0%|          | 0/30 [00:00<?, ?it/s]


L2 — Mejor AUROC (Optuna): 0.8812
   Mejores parámetros: {'C': 8.518710399776031, 'class_weight': 'balanced', 'tfidf_max_features': 100}

Optimizando TARGET: L3 — Intervención crítica en urgencias
Prevalencia train: 0.56%


  0%|          | 0/30 [00:00<?, ?it/s]


L3 — Mejor AUROC (Optuna): 0.9043
   Mejores parámetros: {'C': 2.609403413343495, 'class_weight': 'balanced', 'tfidf_max_features': 93}

Búsqueda completada para todos los targets.


## 5. Resumen de mejores hiperparámetros

La optimización identifica `C` elevados para L3 (menor regularización), consistente con la mayor dificultad de separación en clases muy minoritarias.

In [6]:
df_best = pd.DataFrame(best_params).T
df_best.index.name = 'Target'
df_best['AUROC_optuna'] = pd.Series(best_aurocs)
print('=== Mejores hiperparámetros por target ===\n')
print(df_best.to_string())

=== Mejores hiperparámetros por target ===

               C class_weight tfidf_max_features  AUROC_optuna
Target                                                        
L1      23.71414     balanced                100      0.828092
L2       8.51871     balanced                100      0.881155
L3      2.609403     balanced                 93      0.904343


## 6. Reentrenamiento final con mejores parámetros

Se reentrena con los hiperparámetros óptimos y `max_iter=2000` para asegurar convergencia completa. La función `build_matrix` se invoca con el `max_features` seleccionado por Optuna.

In [7]:
final_models = {}
final_pipelines = {}  # (tfidf, imputer, scaler) por target
final_results = {}

for target in TARGETS:
    y_tr = df_train[target]
    y_v  = df_val[target]
    params = best_params[target].copy()
    max_features = params.pop('tfidf_max_features')

    X_tr, X_v, tfidf = build_matrix(max_features)

    print(f"\n{'='*65}")
    print(f'REENTRENANDO: {target} — {TARGET_NAMES[target]}')
    print(f"{'='*65}")

    # l1_ratio=1 equivale a penalty='l1'; penalty fue deprecado en sklearn 1.8
    clf = LogisticRegression(
        l1_ratio=1,
        solver='liblinear',
        max_iter=2000,        # margen extra en el reentrenamiento final
        random_state=RANDOM_SEED,
        **params,             # C y class_weight
    )
    clf.fit(X_tr, y_tr)

    y_pred = clf.predict_proba(X_v)[:, 1]
    auroc = roc_auc_score(y_v, y_pred)
    auprc = average_precision_score(y_v, y_pred)
    brier = brier_score_loss(y_v, y_pred)

    print(f'AUROC : {auroc:.4f}')
    print(f'AUPRC : {auprc:.4f}  (azar: {y_v.mean():.4f})')
    print(f'Brier : {brier:.4f}')
    print(f'Coeficientes no-cero: {(clf.coef_[0] != 0).sum()} / {clf.coef_[0].size}')

    final_results[target] = {'AUROC': auroc, 'AUPRC': auprc, 'Brier': brier, 'Prev_val': float(y_v.mean())}
    final_models[target]  = clf
    final_pipelines[target] = {'tfidf': tfidf, 'imputer': imputer, 'scaler': scaler, 'max_features': max_features}

print('\nReentrenamiento completado para L1, L2 y L3.')


REENTRENANDO: L1 — Ingreso hospitalario
AUROC : 0.8281
AUPRC : 0.7414  (azar: 0.3831)
Brier : 0.1698
Coeficientes no-cero: 114 / 114

REENTRENANDO: L2 — Resultado crítico en urgencias
AUROC : 0.8812
AUPRC : 0.1270  (azar: 0.0147)
Brier : 0.1422
Coeficientes no-cero: 114 / 114

REENTRENANDO: L3 — Intervención crítica en urgencias
AUROC : 0.9043
AUPRC : 0.1004  (azar: 0.0061)
Brier : 0.1138
Coeficientes no-cero: 106 / 107

Reentrenamiento completado para L1, L2 y L3.


## 7. Resumen de resultados Optuna

Métricas del modelo LogReg optimizado en validación. Las comparativas entre arquitecturas (LogReg, CatBoost, LSTM, TabTransformer, NAM) se realizan en el notebook de evaluación con el test set.

In [8]:
rows = []
for target in TARGETS:
    o = final_results[target]
    rows.append({
        'Target':   target,
        'Nombre':   TARGET_NAMES[target],
        'AUROC':    o['AUROC'],
        'AUPRC':    o['AUPRC'],
        'Brier':    o['Brier'],
        'Prev_val': o['Prev_val'],
    })
df_summary = pd.DataFrame(rows).set_index('Target')
print('=== LogReg + Optuna — Resultados en validación ===')
print(df_summary.round(4).to_string())

=== LogReg + Optuna — Resultados en validación ===
                                   Nombre   AUROC   AUPRC   Brier  Prev_val
Target                                                                     
L1                   Ingreso hospitalario  0.8281  0.7414  0.1698    0.3831
L2         Resultado crítico en urgencias  0.8812  0.1270  0.1422    0.0147
L3      Intervención crítica en urgencias  0.9043  0.1004  0.1138    0.0061


## 8. Guardado de modelos y preprocesadores

Se persisten el modelo LogReg, el vectorizador TF-IDF y los transformadores de escalado e imputación ajustados sobre train, junto con los hiperparámetros óptimos y las métricas finales. Estos artefactos son necesarios para regenerar `predictions_test.parquet` en la fase de evaluación.

In [9]:
TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')
MODELS_DIR = Path('../../models/logreg') / TIMESTAMP
MODELS_DIR.mkdir(parents=True, exist_ok=True)

for target in TARGETS:
    joblib.dump(final_models[target], MODELS_DIR / f'logreg_{target}.joblib')
    joblib.dump(final_pipelines[target], MODELS_DIR / f'pipeline_{target}.joblib')
    print(f'Guardado: logreg_{target}.joblib + pipeline_{target}.joblib')

with open(MODELS_DIR / 'best_params.json', 'w') as f:
    json.dump(best_params, f, indent=2)

with open(MODELS_DIR / 'final_results.json', 'w') as f:
    json.dump(final_results, f, indent=2)

print(f'\nArtefactos guardados en: {MODELS_DIR}')

Guardado: logreg_L1.joblib + pipeline_L1.joblib
Guardado: logreg_L2.joblib + pipeline_L2.joblib
Guardado: logreg_L3.joblib + pipeline_L3.joblib

Artefactos guardados en: ..\..\models\logreg\20260609_211117
